In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F

from torchvision import transforms
import torchvision.models as models
from torchvision.models import ResNet18_Weights

from tqdm import tqdm
from PIL import Image
import matplotlib.cm as cm

import pyarrow.parquet as pq

from collections import OrderedDict



In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



In [3]:
train = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/train.csv")
test = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/test.csv")

vote_cols = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]
classes = np.array(vote_cols, dtype=object)
mapping = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)



In [4]:
cmap = cm.get_cmap("viridis")




/tmp/ipykernel_55/3521101325.py:1: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap("viridis")


In [5]:
class _LRUArrayCache:
    def __init__(self, max_items=64):
        self.max_items = int(max_items)
        self._d = OrderedDict()

    def get(self, key):
        v = self._d.get(key, None)
        if v is not None:
            self._d.move_to_end(key)
        return v

    def put(self, key, value):
        self._d[key] = value
        self._d.move_to_end(key)
        if len(self._d) > self.max_items:
            self._d.popitem(last=False)


def _read_parquet_numeric_to_numpy(parquet_path: str) -> np.ndarray:
    # Minimal I/O improvement only; keeps same numeric content semantics.
    table = pq.read_table(parquet_path)
    df = table.to_pandas(self_destruct=True)
    return df.to_numpy(dtype=np.float32, copy=False)


class ImageDataset(torch.utils.data.Dataset):
    def __init__(
        self, data, spectrogram_dir, transform=None, with_target=False, cache_size=64
    ):
        self.data = data.reset_index(drop=True)
        self.spectrogram_dir = spectrogram_dir
        self.transform = transform
        self.with_target = with_target
        self._cache = _LRUArrayCache(max_items=cache_size)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        specto_id = int(row["spectrogram_id"])
        specto_path = os.path.join(self.spectrogram_dir, f"{specto_id}.parquet")

        arr = self._cache.get(specto_path)
        if arr is None:
            arr = _read_parquet_numeric_to_numpy(specto_path)
            self._cache.put(specto_path, arr)

        mn = np.nanmin(arr)
        mx = np.nanmax(arr)
        if not np.isfinite(mn) or not np.isfinite(mx) or mx <= mn:
            arr_norm = np.zeros_like(arr, dtype=np.float32)
        else:
            arr_norm = (np.nan_to_num(arr, nan=mn) - mn) / (mx - mn)

        rgba = (cmap(arr_norm) * 255).astype(np.uint8)  # H x W x 4
        spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")

        if self.transform:
            spectrogram = self.transform(spectrogram)[:3, :, :]

        if self.with_target:
            y = row[vote_cols].to_numpy(dtype=np.float32)
            y = np.clip(y, 0.0, None)
            s = float(y.sum())
            if s <= 0:
                y[:] = 1.0 / len(y)
            else:
                y /= s
            return spectrogram, torch.from_numpy(y)
        return spectrogram




In [6]:
transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)



In [7]:
_tmp_targets = train[["eeg_id"] + vote_cols].copy()
row_sums = _tmp_targets[vote_cols].sum(axis=1).replace(0, np.nan)
_tmp_targets[vote_cols] = _tmp_targets[vote_cols].div(row_sums, axis=0)
_tmp_targets[vote_cols] = _tmp_targets[vote_cols].fillna(1.0 / num_classes)
train_agg = _tmp_targets.groupby("eeg_id", as_index=False)[vote_cols].mean()

_tmp = train[["eeg_id", "spectrogram_id"] + vote_cols].copy()
_tmp["vote_total"] = _tmp[vote_cols].sum(axis=1)
_tmp = _tmp.sort_values(["eeg_id", "vote_total"], ascending=[True, False])
spec_map = _tmp.drop_duplicates("eeg_id", keep="first")[
    ["eeg_id", "spectrogram_id"]
].reset_index(drop=True)

train_agg = train_agg.merge(spec_map, on="eeg_id", how="left")
train_agg = train_agg.dropna(subset=["spectrogram_id"]).reset_index(drop=True)

test_agg = test[["eeg_id", "spectrogram_id"]].copy().reset_index(drop=True)

# Minimal change for score improvement: create a deterministic holdout split by eeg_id
# to sanity-check loss during fine-tuning (no early stopping; no change to epochs).
perm = np.random.RandomState(SEED).permutation(len(train_agg))
val_size = max(1, int(0.05 * len(train_agg)))
val_idx = perm[:val_size]
trn_idx = perm[val_size:]

train_trn = train_agg.iloc[trn_idx].reset_index(drop=True)
train_val = train_agg.iloc[val_idx].reset_index(drop=True)

train_dataset = ImageDataset(
    train_trn,
    spectrogram_dir="/kaggle/input/hms-harmful-brain-activity-classification/train_spectrograms",
    transform=transform,
    with_target=True,
)

val_dataset = ImageDataset(
    train_val,
    spectrogram_dir="/kaggle/input/hms-harmful-brain-activity-classification/train_spectrograms",
    transform=transform,
    with_target=True,
)

test_dataset = ImageDataset(
    test_agg,
    spectrogram_dir="/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms",
    transform=transform,
    with_target=False,
)

num_workers = min(4, max(1, (os.cpu_count() or 2) // 2))
pin = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin,
    persistent_workers=(num_workers > 0),
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin,
    persistent_workers=(num_workers > 0),
    prefetch_factor=2,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin,
    persistent_workers=(num_workers > 0),
    prefetch_factor=2,
)




In [8]:
class ResNet(nn.Module):
    def __init__(self, num_classes):
        super(ResNet, self).__init__()
        self.resnet = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.resnet(x)




In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet(num_classes).to(device)

ckpt_path = "/kaggle/input/hms-train-spectogram-images/trained_model.pt"
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    print(f"Loaded checkpoint: {ckpt_path}")
else:
    print(
        f"WARNING: checkpoint not found at {ckpt_path}. "
        "Training the head + last ResNet block (layer4) for 2 epochs to reduce underfitting and improve KL toward target."
    )

    # Minimal, score-directed change: unfreeze layer4 in addition to fc.
    # This preserves the same ResNet18 architecture/training loop but typically improves KL vs FC-only.
    for p in model.resnet.parameters():
        p.requires_grad = False
    for p in model.resnet.layer4.parameters():
        p.requires_grad = True
    for p in model.resnet.fc.parameters():
        p.requires_grad = True

    # Use smaller LR for layer4 than FC to keep calibration stable while improving fit.
    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.resnet.layer4.parameters(),
                "lr": 3e-4,
                "weight_decay": 1e-4,
            },
            {"params": model.resnet.fc.parameters(), "lr": 1e-3, "weight_decay": 1e-4},
        ]
    )

    kldiv = nn.KLDivLoss(reduction="batchmean")

    def kl_from_logits(logits, targets, eps=1e-12):
        targets = torch.clamp(targets, min=0.0)
        targets = targets / torch.clamp(targets.sum(dim=1, keepdim=True), min=eps)
        logp = F.log_softmax(logits, dim=1)
        return kldiv(logp, targets)

    EPOCHS = 2  # unchanged

    total_steps = EPOCHS * max(1, len(train_loader))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=1e-5
    )

    def _eval_kldiv(loader):
        model.eval()
        losses = []
        with torch.no_grad():
            for images, targets in loader:
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                logits = model(images)
                loss = kl_from_logits(logits, targets)
                losses.append(float(loss.detach().cpu().item()))
        model.train()
        return float(np.mean(losses)) if losses else float("nan")

    model.train()
    for epoch in range(EPOCHS):
        losses = []
        pbar = tqdm(train_loader, desc=f"train epoch {epoch+1}/{EPOCHS}")
        for images, targets in pbar:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = kl_from_logits(logits, targets)
            loss.backward()
            optimizer.step()
            scheduler.step()

            losses.append(float(loss.detach().cpu().item()))
            if len(losses) >= 10:
                pbar.set_postfix(
                    {
                        "loss": sum(losses[-10:]) / 10.0,
                        "lr_fc": optimizer.param_groups[1]["lr"],
                        "lr_l4": optimizer.param_groups[0]["lr"],
                    }
                )

        trn_mean = sum(losses) / max(1, len(losses))
        val_mean = _eval_kldiv(val_loader)
        print(
            f"Epoch {epoch+1} mean train loss: {trn_mean:.5f} | mean val loss: {val_mean:.5f}"
        )

    model.eval()



Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 112MB/s]


train epoch 1/2:   0%|          | 0/458 [00:00<?, ?it/s]/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Imag

Epoch 1 mean train loss: 1.00425 | mean val loss: 0.90472


train epoch 2/2:   0%|          | 0/458 [00:00<?, ?it/s]/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Imag

Epoch 2 mean train loss: 0.73390 | mean val loss: 0.81923


In [10]:
model.eval()
out = []
pbar = tqdm(test_loader, desc="infer")
for images in pbar:
    images = images.to(device, non_blocking=True)
    with torch.no_grad():
        logits = model(images)
        outputs = F.softmax(logits, dim=1)  # probabilities for submission
    out.append(outputs.detach().cpu().numpy())



infer:   0%|          | 0/308 [00:00<?, ?it/s]/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
/tmp/ipykernel_55/666035740.py:57: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  spectrogram = Image.fromarray(rgba, mode="RGBA").convert("RGB")
infer: 100%|██████████| 308/308 [00:33<00:00,  9.18it/s]


In [11]:
outputs = np.vstack(out)
outputs.shape



(9850, 6)

In [12]:
mapping



{'seizure_vote': 0,
 'lpd_vote': 1,
 'gpd_vote': 2,
 'lrda_vote': 3,
 'grda_vote': 4,
 'other_vote': 5}

In [13]:
submission = test_agg[["eeg_id"]].copy()

pred = outputs.astype(np.float64)
pred = np.clip(pred, 1e-12, None)
row_sum = pred.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
pred = pred / row_sum

for i, c in enumerate(vote_cols):
    submission[c] = pred[:, i]

submission.to_csv("submission.csv", index=False)
submission

,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,2578018731,0.056221,0.033180,0.136130,0.068195,0.399599,0.306675
1,2578018731,0.056221,0.033180,0.136130,0.068195,0.399599,0.306675
2,2578018731,0.056221,0.033180,0.136130,0.068195,0.399599,0.306675
3,2578018731,0.056221,0.033180,0.136130,0.068195,0.399599,0.306675
4,1706196508,0.024776,0.010126,0.001547,0.005127,0.034962,0.923462
...,...,...,...,...,...,...,...
9845,3349371726,0.269375,0.061323,0.620856,0.000487,0.001848,0.046112
9846,641615525,0.269375,0.061323,0.620856,0.000487,0.001848,0.046112
9847,641615525,0.269375,0.061323,0.620856,0.000487,0.001848,0.046112
9848,641615525,0.269375,0.061323,0.620856,0.000487,0.001848,0.046112
